In [16]:
from openai import OpenAI

openai_client= OpenAI()



In [17]:
from gitsource import GithubRepositoryDataReader, chunk_documents
reader= GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md","mdx"}
)
files=reader.read()

parsed_docs=[doc.parse() for doc in files]
chunked_docs= chunk_documents(parsed_docs,size=3000,step= 1500)



In [18]:
from minsearch import AppendableIndex

index=AppendableIndex(
    text_fields=["title","description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)



In [19]:
def search(query):
    results= index.search(
        query=query,
        num_results=5
    )
    return results



In [20]:
#setup rag prompt template
import json
RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context=json.dumps(search_results,indent=2),
    return RAG_PROMPT_TEMPLATE.format(question=question,context=context)




In [21]:
#make a query and build the prompt
question="how do i create a dashbord in evidently"
search_results=search(question)
user_prompt=build_prompt(question, search_results)

#call the LLM
messages=[
    {"role":"system", "content":RAG_INSTRUCTIONS},
    {"role":"user", "content":user_prompt}
]

response=openai_client.responses.create(
    model='gpt-5.4-nano',
    input=messages
)
print (response.output_text)

The provided documentation context doesn’t explain how to create a **dashboard** in Evidently (it mentions that the Evidently Platform offers “rich dashboards”, but there are no steps or UI instructions for creating one).

If you can share the specific documentation page/section you’re looking at (or more context), I can help extract the exact steps.


In [22]:
# lets make it agentic
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-5.4-nano',
    input=messages,
)

response.usage.input_tokens



59

In [23]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response=openai_client.responses.create(
    model='gpt-5.4-nano',
    input=messages,
    tools=[search_tool]
)

response.usage.input_tokens



112

In [24]:
#handling the tool call
tool_call=response.output[0]
tool_call

ResponseFunctionToolCall(arguments='{"query":"create dashboard in evidently how to"}', call_id='call_2NuokCsMR7R1Ln3JIzZgvFqF', name='search', type='function_call', id='fc_0fa2a95e1f8d98f8006ad3620ba5a887d2a5d58466d5651e58', async_=None, caller=None, namespace=None, status='completed')

In [25]:
messages.append(tool_call)
print(tool_call.arguments)
arguments=json.loads(tool_call.arguments)
print(arguments)
#execute the searh with those arguments
#explictly first

search_results=search(**arguments)
search_results[:1]

{"query":"create dashboard in evidently how to"}
{'query': 'create dashboard in evidently how to'}


[{'start': 0,
  'content': "<Check>\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n## What is a Dashboard?\n\nA Dashboard provides a clear view of your AI application performance. You can use it:\n\n- to track evaluation results across multiple experiments;\n- to track live production quality over time.\n\nEach Project has its own Dashboard. It's empty at first.\n\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\n\n![](/images/dashboard_llm_light.png)\n\n## Dashboard Tabs\n\n<Check>\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\nYou can logically organize Panels within the same Dashboard into different Tabs.\n\n![](/images/dashboard_llm_tabs.gif)\n\n## **Dashboard Panels**\n\nA Panel is a visual element in the Dashboard that displays specified values in a single widget. Panels ca

In [26]:
#now we need to call the tool as indicated by AI
call_output= {
    "type":"function_call_output",
    "call_id":tool_call.id,
    "output": json.dumps(search_results)
}
print (call_output)
messages.append(call_output)

#now finall call


{'type': 'function_call_output', 'call_id': 'fc_0fa2a95e1f8d98f8006ad3620ba5a887d2a5d58466d5651e58', 'output': '[{"start": 0, "content": "<Check>\\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\\n</Check>\\n\\n## What is a Dashboard?\\n\\nA Dashboard provides a clear view of your AI application performance. You can use it:\\n\\n- to track evaluation results across multiple experiments;\\n- to track live production quality over time.\\n\\nEach Project has its own Dashboard. It\'s empty at first.\\n\\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\\n\\n![](/images/dashboard_llm_light.png)\\n\\n## Dashboard Tabs\\n\\n<Check>\\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\\n</Check>\\n\\nYou can logically organize Panels within the same Dashboard into different Tabs.\\n\\n![](/images/dashboard_llm_tabs.gif)

In [27]:
response = openai_client.responses.create(
    model='gpt-5.4-nano',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

BadRequestError: Error code: 400 - {'error': {'message': 'No tool output found for function call call_2NuokCsMR7R1Ln3JIzZgvFqF.', 'type': 'invalid_request_error', 'param': 'input', 'code': None}}